# 錄影 → 抽幀（在 JetBot 上跑）

**用途**：工作一（警示物件）補拍。用 JetBot 的相機錄影，再從影片每隔 1~2 秒抽一張 JPG，
檔名與資料夾沿用《影像蒐集工作分配表》第五節（`raw/task_A_alert/alert_地點_時段_0001.jpg`）。

流程：**① 開相機、錄影（原地拍，或「前進並錄影」）→ ② 抽幀 → ③ 收尾釋放相機**

| 項目 | 設定 |
|---|---|
| 檔名 | **原地錄影**：`alert_地點_時段_0001.jpg`（資料夾共用流水號）<br>**前進錄影**：影片、`.json`、抽出的 JPG 都用 `uuid1()` 當檔名（做法同 `data_collection_plus.ipynb`），存在 `raw/task_A_alert/forward/` |
| 解析度 | 640×480（4:3），不裁切不縮放 |
| 錄影 | `REC_FPS = 10` 幀/秒，MJPG 的 `.avi`，存在 `raw/task_A_alert/videos/` |
| 抽幀 | 預設每 1.5 秒一張，存 JPG（品質 95），流水號與同資料夾共用 |
| 紀錄 | 每支影片一個 `.json`（地點、時段、解析度），抽幀會寫 `extraction_log.csv` |

### 拍之前先確認
1. 攝影機角度已固定、曝光／白平衡依第三節鎖定（同一時段全程不改）
2. **SD 卡空間**：MJPG 640×480 約 1~2 MB/秒，**一分鐘約 60~120 MB**。下一格會顯示剩餘空間
3. 機器人留在**人行道**，不進入車道；拍行人與車牌之後若要公開，需遮蔽臉與車牌
4. 目標高度要大於 24 px（640×480 原圖）：每個目標拍近、中、遠
5. **同一支影片只屬於一個資料集切分**（train／valid／test）。test 用的影片請在不同地點或不同日期另外錄
6. 流水號是整個資料夾共用的：換地點、換時段不會歸零

### 前進並錄影（蒐集「車子往前走」的畫面）
機器人實際跟隨時是邊走邊看，所以除了原地錄影，也要錄**前進中**的畫面（會有晃動、接近目標時目標變大）。

- 按「▶ 前進並錄影」：自動開始錄影 → 以設定的速度直行 → 時間到或被擋住就停車，並停止錄影
- **安全條件（務必遵守）**：
  1. 一定要有**一個人在旁邊盯著**，手放在「■ 停止」上，路線事先走過、確認沒有人與車
  2. 只在**人行道或空曠處**拍，不進入車道；速度預設 0.25，上限 0.4，最長 10 秒
  3. 有接 Arduino 雷射測距時，前方距離小於「停車距離」會自動停；**測距不是萬能的**（深色、吸光的表面可能量不到），不能取代人盯著
  4. 沒接測距時預設**不能前進**，要自己勾「允許無測距前進」
- 直行會偏：用「左右修正」微調（正值＝往右偏），先用短距離試走
- 前進錄影的檔名是 `uuid1()`（例如 `3f2c…-….avi`），整個資料夾不會撞名，多台車、多人同時拍也安全；**地點、時段、速度、停車原因都記在同名的 `.json`**，抽幀時 `extraction_log.csv` 會把每張 JPG 的來源影片、地點、時段記下來
- 錄壞了（一開始就被擋住、鏡頭被遮住）按「🗑 刪除上一支前進錄影」，只能刪最近一支，刪完按鈕會變灰
- 抽幀時間隔會自動帶入 1.0 秒

### 遙控沒反應時（按了不動）
先看第一個程式格的輸出：
- `dfrobot_robot.py NOT FOUND`：JetBot 的 notebook 資料夾裡沒有驅動檔。把專案 `src/` 的 `dfrobot_robot.py`（要用測距再加 `feedback.py`、`rangefinder.py`）**上傳到這個 notebook 所在的資料夾**，重跑第一格
- `Remote I/O error`／`OSError`：馬達板沒供電，或 I2C 位址不對。在 JetBot 終端機執行 `sudo i2cdetect -y -r 1`，DFRobot 板應出現在 `10`，官方板在 `60`
- 輸出 `Motor driver: ...` 但還是不動：馬達板要**外接電源**；先把速度拉到 0.4、點動秒數拉到 1.0 再試

### 遙控提醒
- 方向鍵是「**點動**」：按一下只動 `點動秒數` 就自動停，網路斷線也不會一路衝出去
- 錄影時動作放慢，速度建議 0.2~0.3，畫面晃動大的影格抽幀時會被模糊篩掉
- 任何時候都可以按紅色「停止」

### 收尾
拍完、抽完一定要跑最後一格釋放相機，否則其他 notebook 會開不起來。

In [ ]:
# ① 開相機 + 錄影 + 遙控（寫給 Python 3.6 / JetPack 4.x）
import os
import re
import time
import json
import shutil
import threading
from datetime import datetime
from uuid import uuid1

import cv2
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

CAM_W, CAM_H = 640, 480          # 與 YOLO 訓練、上車推論一致
REC_FPS = 10                     # 錄影寫檔幀率（相機實際約 21~30 FPS，這裡降頻省空間與運算）
FRAME_DIR = 'raw/task_A_alert'   # 抽出的 JPG
VIDEO_DIR = FRAME_DIR + '/videos'
# 前進錄影：檔名用 uuid1（同 data_collection_plus.ipynb），與原地錄影的 alert_* 分開放
FORWARD_DIR = FRAME_DIR + '/forward'
FORWARD_VIDEO_DIR = FORWARD_DIR + '/videos'
for _d in (FRAME_DIR, VIDEO_DIR, FORWARD_DIR, FORWARD_VIDEO_DIR):
    if not os.path.isdir(_d):
        os.makedirs(_d)

camera = Camera.instance(width=CAM_W, height=CAM_H)

# --- 馬達板：與 live_demo_following_sm.ipynb 相同的自動選擇 ------------------
# 找不到馬達板也沒關係，只是不能遙控，錄影照常可用。
# （訊息一律英文：容器 locale 不是 UTF-8）
# dfrobot_robot.py / feedback.py / rangefinder.py 在專案的 src/ 裡。JetBot 上的 notebook 資料夾
# 通常沒有它們，所以這裡先把可能的位置加進搜尋路徑；找不到就請把這三個檔案
# 上傳到「這個 notebook 所在的資料夾」（JupyterLab 左側檔案列的上傳鈕）。
import sys
for _p in ('.', 'src', '../src', '../../src', os.path.expanduser('~/src'),
           '/home/jetbot/src', '/home/jetbot/Notebooks', '/home/jetbot/notebooks'):
    if os.path.isfile(os.path.join(_p, 'dfrobot_robot.py')) and os.path.abspath(_p) not in sys.path:
        sys.path.insert(0, os.path.abspath(_p))
        print('src path added:', os.path.abspath(_p))

robot = None
try:
    from dfrobot_robot import Robot
    robot = Robot()
    print('Motor driver: DFRobot DFR0592 (0x10)')
except Exception as _e:
    if isinstance(_e, ImportError):
        print('dfrobot_robot.py NOT FOUND. Upload src/dfrobot_robot.py next to this notebook, then rerun.')
    else:
        print('DFRobot driver failed: %s: %s' % (type(_e).__name__, _e))
    print('Trying official jetbot.Robot (it fails with OSError 121 on a DFRobot board) ...')
    try:
        from jetbot import Robot
        robot = Robot()
        print('Motor driver: Adafruit MotorHAT (0x60)')
    except Exception as _e2:
        print('No motor driver (%s). Recording works, remote control disabled.' % type(_e2).__name__)

# --- 雷射測距（Arduino VL53L0X，選配）：前進時用來自動停車 -----------------
# feedback.Feedback() 會獨佔序列埠 /dev/ttyACM0，所以收尾那一格一定要關掉它。
# 失敗不影響錄影與點動遙控，只是「前進並錄影」預設不開放。
fb = None
finder = None
try:
    import feedback
    import rangefinder
    fb = feedback.Feedback()
    finder = rangefinder.RangeFinder(fb)
    print('Laser rangefinder backend:', fb.backend.name)
except Exception as _e3:
    print('No rangefinder (%s). Forward-and-record needs the override checkbox.' % type(_e3).__name__)


# --- 錄影器 ------------------------------------------------------------------
class Recorder(object):
    """掛在相機的 value 變化上，依 REC_FPS 降頻後寫進 VideoWriter。"""

    def __init__(self):
        self.writer = None
        self.lock = threading.Lock()
        self.last_t = 0.0
        self.frames = 0
        self.started = None
        self.path = None

    @property
    def recording(self):
        return self.writer is not None

    def start(self, path):
        writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MJPG'), REC_FPS, (CAM_W, CAM_H))
        if not writer.isOpened():
            raise RuntimeError('VideoWriter 開不起來：%s' % path)
        with self.lock:
            self.writer, self.path = writer, path
            self.frames, self.last_t, self.started = 0, 0.0, time.time()

    def on_frame(self, change):
        if self.writer is None:
            return
        now = time.time()
        if now - self.last_t < 1.0 / REC_FPS:
            return
        self.last_t = now
        with self.lock:
            if self.writer is not None:
                self.writer.write(change['new'])
                self.frames += 1

    def stop(self):
        with self.lock:
            writer, self.writer = self.writer, None
        if writer is not None:
            writer.release()
        return self.path, self.frames


rec = Recorder()
camera.observe(rec.on_frame, names='value')

# --- 畫面 + 控制項 -----------------------------------------------------------
image_widget = widgets.Image(format='jpeg', width=480, height=360)
camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

loc_widget = widgets.Text(value='road', description='地點',
                          placeholder='gate / road / bg', layout=widgets.Layout(width='260px'))
slot_widget = widgets.Dropdown(options=['am', 'noon', 'pm'], value='am', description='時段',
                               layout=widgets.Layout(width='260px'))
note_widget = widgets.Text(value='', description='備註',
                           placeholder='例如：公車站、陰天、輛多', layout=widgets.Layout(width='260px'))
hint = widgets.HTML('<small>地點：gate(校門口) / road(臨溪路) / bg(純背景)；不能有底線或空白</small>')
rec_button = widgets.Button(description='● 開始錄影', button_style='success',
                            layout=widgets.Layout(width='140px', height='56px'))
status = widgets.HTML()

speed_widget = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05, description='速度',
                                   layout=widgets.Layout(width='260px'))
pulse_widget = widgets.FloatSlider(value=0.4, min=0.1, max=1.5, step=0.1, description='點動秒數',
                                   layout=widgets.Layout(width='260px'))
_bl = widgets.Layout(width='64px', height='44px')
fwd_b = widgets.Button(description='↑', layout=_bl)
back_b = widgets.Button(description='↓', layout=_bl)
left_b = widgets.Button(description='←', layout=_bl)
right_b = widgets.Button(description='→', layout=_bl)
stop_b = widgets.Button(description='停止', button_style='danger', layout=_bl)
for _b in (fwd_b, back_b, left_b, right_b, stop_b):
    _b.disabled = robot is None


# --- 前進並錄影 ---------------------------------------------------------------
fwd_speed_w = widgets.FloatSlider(value=0.25, min=0.1, max=0.4, step=0.05, description='前進速度',
                                  layout=widgets.Layout(width='260px'))
fwd_secs_w = widgets.FloatSlider(value=5.0, min=1.0, max=10.0, step=0.5, description='前進秒數',
                                 layout=widgets.Layout(width='260px'))
trim_w = widgets.FloatSlider(value=0.0, min=-0.1, max=0.1, step=0.01, description='左右修正',
                             layout=widgets.Layout(width='260px'))
stop_dist_w = widgets.FloatSlider(value=0.5, min=0.3, max=2.0, step=0.1, description='停車距離(m)',
                                  layout=widgets.Layout(width='260px'))
force_w = widgets.Checkbox(value=False, description='允許無測距前進（需有人在旁邊盯著）', indent=False,
                           layout=widgets.Layout(width='320px'))
fwd_go = widgets.Button(description='▶ 前進並錄影', button_style='warning',
                        layout=widgets.Layout(width='160px', height='48px'))
fwd_stop = widgets.Button(description='■ 停止', button_style='danger',
                          layout=widgets.Layout(width='100px', height='48px'))
fwd_info = widgets.HTML()
fwd_undo = widgets.Button(description='🗑 刪除上一支前進錄影', button_style='warning',
                          layout=widgets.Layout(width='260px'), disabled=True)
_last_fwd = {'stem': None}
fwd = {'running': False, 'abort': threading.Event()}
MAX_FWD_SPEED, MAX_FWD_SECS = 0.4, 10.0   # 硬上限，不看滑桿

_timer = {'t': None}


def _stop_motors(*_):
    if _timer['t'] is not None:
        _timer['t'].cancel()
        _timer['t'] = None
    if robot is not None:
        robot.stop()


def _pulse(move):
    """動一小段就自動停。連按會取消前一個計時器，重新計時。"""
    if robot is None:
        return
    if _timer['t'] is not None:
        _timer['t'].cancel()
    move(float(speed_widget.value))
    _timer['t'] = threading.Timer(float(pulse_widget.value), robot.stop)
    _timer['t'].start()


fwd_b.on_click(lambda _: _pulse(robot.forward))
back_b.on_click(lambda _: _pulse(robot.backward))
left_b.on_click(lambda _: _pulse(robot.left))
right_b.on_click(lambda _: _pulse(robot.right))


def _distance():
    """雷射距離（公尺）。沒有讀數（沒接、超出量程、表面吸光）回 None。"""
    try:
        return finder.distance if finder is not None else None
    except Exception:
        return None


def _forward_worker(speed, secs, trim, stop_dist, meta_path, started_recording):
    reason = 'time'
    t0 = time.time()
    try:
        robot.set_motors(speed + trim, speed - trim)
        while time.time() - t0 < secs:
            if fwd['abort'].is_set():
                reason = 'manual'
                break
            d = _distance()
            if d is not None and d < stop_dist:
                reason = 'blocked(%.2fm)' % d
                break
            time.sleep(0.05)
    except Exception as e:                      # 任何錯誤都先停車
        reason = 'error:%s' % type(e).__name__
    finally:
        robot.stop()
        moved = time.time() - t0
        path, frames = (None, 0)
        if started_recording:
            path, frames = stop_recording({'stopped_by': reason, 'moved_sec': round(moved, 2)}, meta_path)
        if path:
            _last_fwd['stem'] = os.path.splitext(os.path.basename(path))[0]
            fwd_undo.disabled = False
        fwd['running'] = False
        fwd['abort'].clear()
        fwd_go.disabled = False
        rec_button.disabled = False
        for _b in (fwd_b, back_b, left_b, right_b):
            _b.disabled = robot is None
        _refresh('🏁 前進結束（%s，%.1f 秒）%s' % (
            reason, moved, '，存好 <b>%s</b>（%d 幀）' % (os.path.basename(path), frames) if path else ''))


def start_forward(_):
    if robot is None:
        fwd_info.value = '<b style="color:#c00">沒有馬達板，不能前進</b>'
        return
    if fwd['running'] or rec.recording:
        fwd_info.value = '<b style="color:#c00">已在錄影／前進中，先停止</b>'
        return
    if finder is None and not force_w.value:
        fwd_info.value = '<b style="color:#c00">沒有測距：要前進請先勾「允許無測距前進」</b>'
        return
    d0 = _distance()
    if d0 is not None and d0 < float(stop_dist_w.value):
        fwd_info.value = '<b style="color:#c00">前方只有 %.2f m，比停車距離近，不前進</b>' % d0
        return
    speed = min(float(fwd_speed_w.value), MAX_FWD_SPEED)
    secs = min(float(fwd_secs_w.value), MAX_FWD_SECS)
    try:
        stem, meta_path = start_recording({'mode': 'forward', 'speed': speed, 'trim': float(trim_w.value),
                                          'planned_sec': secs, 'stop_dist_m': float(stop_dist_w.value),
                                          'laser': finder is not None})
    except (ValueError, RuntimeError) as e:
        fwd_info.value = '<b style="color:#c00">沒開始：%s</b>' % e
        return
    _stop_motors()
    _last_fwd['stem'] = None
    fwd_undo.disabled = True
    fwd['running'] = True
    fwd['abort'].clear()
    fwd_go.disabled = True
    rec_button.disabled = True
    for _b in (fwd_b, back_b, left_b, right_b):
        _b.disabled = True
    fwd_info.value = ''
    _refresh('▶ 前進中，錄影 <b>%s.avi</b>' % stem)
    threading.Thread(target=_forward_worker,
                     args=(speed, secs, float(trim_w.value), float(stop_dist_w.value), meta_path, True)).start()


def abort_forward(_):
    fwd['abort'].set()          # 前進中：讓 worker 停車並收尾
    _stop_motors()


def undo_forward(_):
    """刪掉最近一支前進錄影（影片 + .json）。刪完清成 None，連按兩次不會誤刪上上一支。"""
    stem = _last_fwd['stem']
    if stem is None or fwd['running']:
        return
    for ext in ('.avi', '.json'):
        p = os.path.join(FORWARD_VIDEO_DIR, stem + ext)
        if os.path.exists(p):
            os.remove(p)
    _last_fwd['stem'] = None
    fwd_undo.disabled = True
    _refresh('🗑️ 已刪除前進錄影 <span style="color:#888">%s</span>' % stem)


fwd_go.on_click(start_forward)
fwd_undo.on_click(undo_forward)
fwd_stop.on_click(abort_forward)
stop_b.on_click(abort_forward)   # 點動區的「停止」也能中止前進


def _clean(widget, name):
    v = (widget.value or '').strip()
    if not v:
        raise ValueError('%s 不能空白' % name)
    if re.search(r'[\s_]', v):
        raise ValueError('%s 不能有底線或空白：「%s」' % (name, v))
    return v


def _refresh(msg=''):
    free_gb = shutil.disk_usage(VIDEO_DIR).free / 1e9
    warn = ' <b style="color:#c00">空間不足！</b>' if free_gb < 1.0 else ''
    if rec.recording:
        sec = time.time() - rec.started
        info = '<b style="color:#c00">錄影中</b> %02d:%02d · %d 幀' % (sec // 60, sec % 60, rec.frames)
    else:
        info = '待機'
    n_videos = len([f for d_ in (VIDEO_DIR, FORWARD_VIDEO_DIR) for f in os.listdir(d_) if f.endswith('.avi')])
    d = _distance()
    dist = '前方 %.2f m' % d if d is not None else ('測距：無讀數' if finder is not None else '測距：未連接')
    status.value = ('%s<br>%s · %s · 影片共 %d 支 · SD 卡剩 %.2f GB%s'
                    % (msg, info, dist, n_videos, free_gb, warn))


def start_recording(extra=None):
    """開始錄影並寫 .json。欄位不合法或 VideoWriter 開不起來會丟例外。回傳 (stem, json 路徑)。"""
    loc = _clean(loc_widget, '地點')
    stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    mode = (extra or {}).get('mode', 'static')
    if mode == 'forward':
        # 前進錄影：uuid1 檔名（時間 + 機器位址，不會撞名），地點與時段只記在 .json
        stem, folder = str(uuid1()), FORWARD_VIDEO_DIR
    else:
        stem, folder = '%s-%s-%s' % (loc, slot_widget.value, stamp), VIDEO_DIR
    path = os.path.join(folder, stem + '.avi')
    rec.start(path)
    meta = {'video': stem + '.avi', 'location': loc, 'slot': slot_widget.value,
            'note': note_widget.value, 'width': CAM_W, 'height': CAM_H,
            'rec_fps': REC_FPS, 'started': datetime.now().isoformat(), 'mode': 'static'}
    meta.update(extra or {})
    meta_path = os.path.join(folder, stem + '.json')
    with open(meta_path, 'w') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    return stem, meta_path


def stop_recording(extra=None, meta_path=None):
    path, frames = rec.stop()
    if meta_path and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        meta['frames'] = frames
        meta.update(extra or {})
        with open(meta_path, 'w') as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
    return path, frames


_static = {'meta': None}


def toggle_record(_):
    if fwd['running']:
        _refresh('<b style="color:#c00">前進中，先按「■ 停止」</b>')
        return
    if not rec.recording:
        try:
            stem, _static['meta'] = start_recording()
        except (ValueError, RuntimeError) as e:
            _refresh('<b style="color:#c00">沒開始：%s</b>' % e)
            return
        rec_button.description, rec_button.button_style = '■ 停止錄影', 'danger'
        _refresh('開始錄影 <b>%s.avi</b>' % stem)
    else:
        path, frames = stop_recording(meta_path=_static['meta'])
        rec_button.description, rec_button.button_style = '● 開始錄影', 'success'
        _refresh('✅ 存好 <b>%s</b>（%d 幀）' % (os.path.basename(path), frames))


rec_button.on_click(toggle_record)


def _ticker():
    while True:
        time.sleep(1.0)
        if rec.recording:
            _refresh()


threading.Thread(target=_ticker, daemon=True).start()
_refresh()

controls = widgets.VBox([
    loc_widget, slot_widget, note_widget, hint, rec_button, status,
    widgets.HTML('<b>遙控（點動）</b>'), speed_widget, pulse_widget,
    widgets.HBox([widgets.Label(layout=_bl), fwd_b, widgets.Label(layout=_bl)]),
    widgets.HBox([left_b, stop_b, right_b]),
    widgets.HBox([widgets.Label(layout=_bl), back_b, widgets.Label(layout=_bl)]),
    widgets.HTML('<b>前進並錄影</b>　<small>要有人在旁邊，手放在停止鍵</small>'),
    fwd_speed_w, fwd_secs_w, trim_w, stop_dist_w, force_w,
    widgets.HBox([fwd_go, fwd_stop]), fwd_undo, fwd_info,
], layout=widgets.Layout(padding='0 0 0 12px'))
# align_items='flex-start'：左右兩欄各自保持原本高度。預設是 stretch，控制欄一長，
# 即時畫面就會被拉成直式（只是顯示變形，錄影與存檔仍是 640x480）。
image_widget.layout = widgets.Layout(width='480px', height='360px')
display(widgets.HBox([image_widget, controls], layout=widgets.Layout(align_items='flex-start')))

### ② 抽幀

選一支錄好的影片，設定抽幀間隔，按「開始抽幀」。

- **間隔**：預設 1.5 秒。連續畫面太像，不必每幀都取
- **模糊篩選**：用 Laplacian 變異數判斷，低於門檻的影格跳過（0 = 不篩）。預設 50 只是起點，先用「預覽」看會跳過幾張再調
- 抽出的影格接在資料夾現有最大流水號後面；**每張的來源影片與時間點記在 `extraction_log.csv`**
- 抽錯可以按「刪除本次抽出的影格」，只會刪剛剛那一批
- 影片在**錄影停止後**才能抽

In [ ]:
# ② 抽幀的函式（只依賴 cv2，可以單獨在電腦上用）
import csv


def next_serial(folder):
    """資料夾共用流水號：取現有 alert_*_NNNN.jpg 的最大 4 位數 + 1。"""
    nums = []
    for n in os.listdir(folder):
        m = re.match(r'^alert_[^_]+_[^_]+_(\d{4})\.jpe?g$', n, re.I)
        if m:
            nums.append(int(m.group(1)))
    return max(nums) + 1 if nums else 1


def blur_score(img):
    return cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()


def plan_extract(video_path, every_sec):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or REC_FPS
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    step = max(1, int(round(every_sec * fps)))
    return {'fps': fps, 'total': total, 'step': step,
            'count': (total + step - 1) // step, 'size': (w, h)}


def extract_frames(video_path, loc, slot, every_sec=1.5, blur_min=0,
                   out_dir=None, quality=95, dry_run=False, uuid_names=False):
    """每 every_sec 秒抽一張 JPG。回傳 (存檔清單, 跳過張數)。

    uuid_names=False（原地錄影）：alert_地點_時段_流水號.jpg，流水號與資料夾共用。
    uuid_names=True（前進錄影）：<uuid1>.jpg，做法同 data_collection_plus.ipynb；
        地點、時段不在檔名裡，記在 extraction_log.csv。
    dry_run=True 只計算「模糊篩選後會存幾張」，不寫檔。
    """
    out_dir = out_dir or FRAME_DIR
    for name, v in (('地點', loc), ('時段', slot)):
        if not v or re.search(r'[\s_]', v):
            raise ValueError('%s 不能空白，也不能有底線或空白：「%s」' % (name, v))
    plan = plan_extract(video_path, every_sec)
    if plan['size'] != (CAM_W, CAM_H):
        print('⚠️ 影片解析度是 %dx%d，不是 %dx%d' % (plan['size'][0], plan['size'][1], CAM_W, CAM_H))
    cap = cv2.VideoCapture(video_path)
    serial = 1 if uuid_names else next_serial(out_dir)
    saved, skipped, idx = [], 0, 0
    rows = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % plan['step'] == 0:
            score = blur_score(frame)
            if blur_min and score < blur_min:
                skipped += 1
            else:
                if uuid_names:
                    name = str(uuid1()) + '.jpg'
                else:
                    if serial > 9999:
                        cap.release()
                        raise RuntimeError('流水號超過 9999，請先跟技術負責人確認命名規則')
                    name = 'alert_%s_%s_%04d.jpg' % (loc, slot, serial)
                    serial += 1
                if not dry_run:
                    cv2.imwrite(os.path.join(out_dir, name), frame,
                                [cv2.IMWRITE_JPEG_QUALITY, quality])
                rows.append([name, os.path.basename(video_path), idx,
                             round(idx / plan['fps'], 2), round(score, 1), loc, slot])
                saved.append(name)
        idx += 1
    cap.release()
    if rows and not dry_run:
        log = os.path.join(out_dir, 'extraction_log.csv')
        new = not os.path.exists(log)
        with open(log, 'a') as f:
            w = csv.writer(f)
            if new:
                w.writerow(['filename', 'source_video', 'frame_idx', 'time_s', 'blur_score',
                            'location', 'slot'])
            w.writerows(rows)
    return saved, skipped


def delete_extracted(names, out_dir=None):
    """刪掉指定的影格，並把它們從 extraction_log.csv 移除。"""
    out_dir = out_dir or FRAME_DIR
    for n in names:
        p = os.path.join(out_dir, n)
        if os.path.exists(p):
            os.remove(p)
    log = os.path.join(out_dir, 'extraction_log.csv')
    if os.path.exists(log):
        with open(log) as f:
            rows = list(csv.reader(f))
        gone = set(names)
        keep = [r for i, r in enumerate(rows) if i == 0 or r[0] not in gone]
        with open(log, 'w') as f:
            csv.writer(f).writerows(keep)

In [ ]:
# ② 抽幀的介面
def _list_videos():
    """回傳 [(顯示文字, 完整路徑)]。前進錄影的檔名是 uuid，所以顯示時帶上地點與時段。"""
    items = []
    for folder, tag in ((VIDEO_DIR, '原地'), (FORWARD_VIDEO_DIR, '前進')):
        for f in os.listdir(folder):
            if not f.endswith('.avi'):
                continue
            label = '[%s] %s' % (tag, f)
            meta = os.path.join(folder, os.path.splitext(f)[0] + '.json')
            if tag == '前進' and os.path.exists(meta):
                try:
                    with open(meta) as fh:
                        m = json.load(fh)
                    label = '[前進] %s %s %s…' % (m.get('location', '?'), m.get('slot', '?'), f[:8])
                except Exception:
                    pass
            items.append((label, os.path.join(folder, f)))
    return sorted(items, key=lambda x: os.path.getmtime(x[1]), reverse=True)


video_dd = widgets.Dropdown(options=_list_videos(), description='影片',
                            layout=widgets.Layout(width='420px'))
refresh_b = widgets.Button(description='重新整理清單', layout=widgets.Layout(width='130px'))
ex_loc = widgets.Text(description='地點', layout=widgets.Layout(width='260px'))
ex_slot = widgets.Dropdown(options=['am', 'noon', 'pm'], description='時段',
                           layout=widgets.Layout(width='260px'))
every_w = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='間隔(秒)',
                              layout=widgets.Layout(width='360px'))
blur_w = widgets.IntText(value=50, description='模糊門檻', layout=widgets.Layout(width='200px'))
preview_b = widgets.Button(description='預覽（不存檔）', layout=widgets.Layout(width='140px'))
go_b = widgets.Button(description='開始抽幀', button_style='success',
                      layout=widgets.Layout(width='140px'))
undo_b = widgets.Button(description='刪除本次抽出的影格', button_style='danger',
                        layout=widgets.Layout(width='180px'), disabled=True)
ex_status = widgets.HTML()
thumbs = widgets.HBox()
_last_batch = {'names': [], 'dir': FRAME_DIR}


def _load_meta(_change=None):
    """選影片時，用旁邊的 .json 自動帶入地點與時段。"""
    if not video_dd.value:
        return
    meta = os.path.splitext(video_dd.value)[0] + '.json'
    if os.path.exists(meta):
        with open(meta) as f:
            m = json.load(f)
        ex_loc.value = m.get('location', ex_loc.value)
        if m.get('slot') in ex_slot.options:
            ex_slot.value = m['slot']
        every_w.value = 1.0 if m.get('mode') == 'forward' else 1.5   # 前進畫面變化快，間隔縮短


def _refresh_videos(_=None):
    video_dd.options = _list_videos()
    _load_meta()


def _selected():
    if rec.recording:
        raise RuntimeError('還在錄影中，先停止錄影')
    if not video_dd.value:
        raise RuntimeError('沒有影片')
    return video_dd.value


def _target(path):
    """前進錄影 -> (forward 資料夾, uuid 檔名)；原地錄影 -> (alert 資料夾, 流水號檔名)。"""
    if os.path.abspath(path).startswith(os.path.abspath(FORWARD_VIDEO_DIR)):
        return FORWARD_DIR, True
    return FRAME_DIR, False


def do_preview(_):
    try:
        path = _selected()
        out_dir, use_uuid = _target(path)
        saved, skipped = extract_frames(path, ex_loc.value.strip(), ex_slot.value,
                                        every_w.value, blur_w.value, out_dir=out_dir,
                                        dry_run=True, uuid_names=use_uuid)
        p = plan_extract(path, every_w.value)
        tail = '檔名用 uuid1' if use_uuid else '起始流水號 %04d' % next_serial(out_dir)
        ex_status.value = ('預覽：影片 %d 幀（%.1f FPS），每 %d 幀取一張 → 預計存 <b>%d</b> 張，'
                           '模糊跳過 %d 張。%s'
                           % (p['total'], p['fps'], p['step'], len(saved), skipped, tail))
    except Exception as e:
        ex_status.value = '<b style="color:#c00">%s</b>' % e


def do_extract(_):
    try:
        path = _selected()
        out_dir, use_uuid = _target(path)
        saved, skipped = extract_frames(path, ex_loc.value.strip(), ex_slot.value,
                                        every_w.value, blur_w.value, out_dir=out_dir,
                                        uuid_names=use_uuid)
    except Exception as e:
        ex_status.value = '<b style="color:#c00">%s</b>' % e
        return
    _last_batch['names'], _last_batch['dir'] = saved, out_dir
    undo_b.disabled = not saved
    n_jpg = len([f for f in os.listdir(out_dir) if f.lower().endswith('.jpg')])
    free_gb = shutil.disk_usage(out_dir).free / 1e9
    ex_status.value = ('✅ 抽出 <b>%d</b> 張（模糊跳過 %d 張）：%s ~ %s<br>'
                       '%s/ 共 %d 張 · SD 卡剩 %.2f GB'
                       % (len(saved), skipped, saved[0] if saved else '-',
                          saved[-1] if saved else '-', out_dir, n_jpg, free_gb))
    items = []
    for n in saved[:6]:
        img = cv2.imread(os.path.join(out_dir, n))
        small = cv2.resize(img, (160, 120))
        items.append(widgets.Image(value=bgr8_to_jpeg(small), format='jpeg', width=160, height=120))
    thumbs.children = items


def do_undo(_):
    names = _last_batch['names']
    delete_extracted(names, _last_batch['dir'])
    ex_status.value = '🗑️ 已刪除本次抽出的 %d 張' % len(names)
    _last_batch['names'] = []
    undo_b.disabled = True
    thumbs.children = []


refresh_b.on_click(_refresh_videos)
video_dd.observe(_load_meta, names='value')
preview_b.on_click(do_preview)
go_b.on_click(do_extract)
undo_b.on_click(do_undo)
_load_meta()

display(widgets.VBox([
    widgets.HBox([video_dd, refresh_b]),
    widgets.HBox([ex_loc, ex_slot]),
    widgets.HBox([every_w, blur_w]),
    widgets.HBox([preview_b, go_b, undo_b]),
    ex_status, thumbs,
]))

### ③ 收尾

拍完、抽完要跑下面這格：停止馬達、停止錄影、釋放相機。
萬一忘了跑就關掉 notebook，到 Kernel → Restart，或重開 JetBot 的 Jupyter。

抽出的 JPG 之後上傳 Roboflow 前，請：
1. 瀏覽 `raw/task_A_alert/` 與 `raw/task_A_alert/forward/`，刪掉沒有目標又不打算當負樣本的影格、重複或嚴重模糊的影格
2. 把同一支影片的影格視為同一組，切分時整組放進同一個資料集（來源見 `extraction_log.csv`）
3. 備份兩個 `videos/`（原始影片）與 `raw/task_A_alert/`（含 `forward/`）

In [ ]:
# ③ 收尾
fwd['abort'].set()
_stop_motors()
if rec.recording:
    path, frames = rec.stop()
    print('已停止錄影：%s（%d 幀）' % (path, frames))
camera.unobserve(rec.on_frame, names='value')
camera_link.unlink()
camera.stop()
if fb is not None:
    fb.close()      # 釋放 Arduino 序列埠（不關的話下次會「Device or resource busy」）
print('相機已釋放。要繼續拍請從第一個程式格重跑。')